In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import nltk
from nltk.tokenize import word_tokenize
from nltk import download
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier


In [ ]:
data = pd.read_csv("twitter_training.csv", names=['id', 'game', 'sentiment', 'text'])


In [ ]:
data.head(100)

In [ ]:
data.describe()

In [ ]:
data.drop_duplicates( inplace=True)
data.dropna(inplace=True)
data = data[~data['sentiment'].str.contains('Irrelevant', na=False)]

In [ ]:
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

In [ ]:
def preprocess_text(text):
    text = str(text).lower()

    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)

    text = re.sub(r'[^\w\s]', '', text)

    text = re.sub(r'\d+', '', text)

    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(word) for word in tokens]

    return ' '.join(stemmed_tokens)

data['clreview'] = data['text'].apply(preprocess_text)

In [ ]:
data['clreview'].head(10)

In [ ]:
data['sentiment'].values

In [ ]:
X = data['clreview']
lable = { 'Negative':-1,'Positive' :1 ,'Neutral':0}
data['sentiment'] = data['sentiment'].map(lable)
y = data['sentiment']

In [ ]:
X

In [ ]:
y

In [ ]:
vectorizer = TfidfVectorizer()
X_v = vectorizer.fit_transform(X)


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X_v, y, test_size=0.2, random_state=42)


In [ ]:
print("Vectorized x_train shape:", x_train.shape)
print("Vectorized x_test shape:", x_test.shape)

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(x_train, y_train)


In [ ]:
y_pred_nb = nb_model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred_nb)
print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
conf_matrix = confusion_matrix(y_test, y_pred_nb)

fig, ax = plt.subplots()
cax = ax.matshow(conf_matrix, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

fig.colorbar(cax)

for (i, j), val in np.ndenumerate(conf_matrix):
    ax.text(j, i, f'{val}', ha='center', va='center', color='black')

ax.set_xticklabels([''] + ['Negative', 'Positive','Neutral'])
ax.set_yticklabels([''] + ['Negative', 'Positive','Neutral'])

plt.show()

In [ ]:
def predict_sentiment(texts):
    label_to_sentiment = {-1: 'Negative', 1: 'Positive',0:'Neutral'}
    texts = [preprocess_text(text) for text in texts]
    transformed_texts = vectorizer.transform(texts)
    predictions = nb_model.predict(transformed_texts)
    predicted_labels = [label_to_sentiment[pred] for pred in predictions]
    for text, label in zip(texts, predicted_labels):
        print(f'Text: {text} --> Sentiment: {label}')

user_input = input("Please enter your text: ")
predict_sentiment([user_input])